# 44 · k-means com k = 3 e oito variáveis

Pergunta: entre os assinantes ativos em 30/09/2026 que treinaram nos 90 dias até o corte, em quantos
grupos eles se dividem, o que distingue cada grupo e o que a Vitaliza faria diferente com cada um?

**k = 3** é escolha do grupo. As oito variáveis são as quatro da aula 3 mais quatro do grupo:

| Variável | Coluna | Tratamento |
|---|---|---|
| Treinos por semana | `treinos_por_semana` | z-score |
| Dias sem treino | `dias_sem_treinar_log` | log(1 + x) do notebook 41, z-score |
| Tempo de casa | `duracao_dias_log` | log(1 + x) do notebook 41, z-score |
| % dos treinos ao ar livre | nova: (corrida + caminhada + pedal) ÷ treinos na janela | z-score |
| Km por treino | nova, de `atividades_limpa.parquet` | log(1 + x), z-score; 0 para quem não tem treino com distância |
| Engajamento | `nivel_engajamento` | ordinal (possível churn 0 → alto 3), z-score |
| Plano | `plano` | indicador 0/1 (anual = 1), sem padronizar |
| Canal de aquisição | `canal_aquisicao` | um indicador 0/1 por canal (6), sem padronizar |

As numéricas são padronizadas de novo entre os elegíveis (média 0 e desvio 1 nos 13.999 que entram),
e os indicadores seguem a regra do notebook 42: não são padronizados.

Algoritmo: `KMeans(n_clusters=3, n_init=10, random_state=0)`. Avaliação pelas quatro lentes da aula:
inércia, silhueta, estabilidade (índice de Rand ajustado em reamostragens) e leitura de negócio.
Nada em `dados/brutos/` é lido nem alterado. Ordem de execução: 39 → 41 → 42 → 43 → 44.

In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.preprocessing import StandardScaler

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
P = Path("../dados/processados"); S = Path("../saidas")
CORTE = pd.Timestamp("2026-09-30")
K = 3; SEED = 0
AR_LIVRE = ["corrida", "caminhada", "pedal"]

m = pd.read_parquet(P / "tabela_master.parquet")
print("tabela master:", m.shape)
print(m.segmento_regra.value_counts().to_string())
e = m[m.segmento_regra == "elegível ao k-means"].copy().reset_index(drop=True)
assert (e.data_referencia == CORTE).all() and e.ativa_no_corte.all()
print("\nelegíveis ao k-means:", len(e), "| contas distintas:", e.user_id.nunique())

tabela master: (63929, 119)
segmento_regra
fora do escopo (não é assinante ativo)    49176
elegível ao k-means                       13999
inativo em 90 dias                          754

elegíveis ao k-means: 13999 | contas distintas: 13999


## 1. % dos treinos ao ar livre

In [2]:
e["pct_ar_livre"] = (e[["treinos_" + t for t in AR_LIVRE]].sum(axis=1) / e.treinos_90d * 100).astype(float)
print("treinos_90d > 0 em todos os elegíveis:", bool((e.treinos_90d > 0).all()))
print(e.pct_ar_livre.describe().round(1).to_dict())
print("0% ao ar livre:", int((e.pct_ar_livre == 0).sum()), "| 100% ao ar livre:", int((e.pct_ar_livre == 100).sum()))

treinos_90d > 0 em todos os elegíveis: True
{'count': 13999.0, 'mean': 54.4, 'std': 34.6, 'min': 0.0, '25%': 16.7, '50%': 71.4, '75%': 83.3, 'max': 100.0}
0% ao ar livre: 691 | 100% ao ar livre: 1087


## 2. Km por treino

Média de `distancia_km` dos treinos ao ar livre (corrida, caminhada, pedal) na janela de 90 dias até o
corte (mesma janela da tabela master: 0 a 89 dias antes de 30/09/2026), só treinos com GPS válido e
distância maior que zero. Quem não tem nenhum treino com distância na janela fica com **0 km**.

In [3]:
a = pd.read_parquet(P / "atividades_limpa.parquet", columns=["user_id", "data", "tipo", "distancia_km", "gps_ok"])
print("treinos no arquivo:", len(a))
a = a[a.user_id.isin(e.user_id)]
print("-> dos elegíveis:", len(a))
a = a[(CORTE - a.data).dt.days.between(0, 89)]
print("-> na janela de 90 dias:", len(a))
al = a[a.tipo.isin(AR_LIVRE)]
print("-> ao ar livre:", len(al), "| GPS falhou:", int((~al.gps_ok).sum()), "| GPS ok sem distância ou zero:", int((al.gps_ok & ~(al.distancia_km > 0)).sum()))
d = al[al.gps_ok & (al.distancia_km > 0)]
print("-> com distância válida:", len(d), "| contas:", d.user_id.nunique())
km = d.groupby("user_id").distancia_km.mean().rename("km_por_treino")
e = e.merge(km, on="user_id", how="left")
fal = set(al.user_id) - set(d.user_id)
print("\ncontas sem treino com distância (ficam com 0 km):", int(e.km_por_treino.isna().sum()))
print("  das quais têm treino ao ar livre na janela, todos sem distância válida (GPS falhou):", len(fal))
e["km_por_treino"] = e.km_por_treino.fillna(0)
e["km_por_treino_log"] = np.log1p(e.km_por_treino)
print("\nkm por treino, quem tem distância:", e.km_por_treino[e.km_por_treino > 0].describe().round(1).to_dict())
print("km por treino por tipo principal (mediana, quem tem distância):")
print(e[e.km_por_treino > 0].groupby("tipo_principal").km_por_treino.median().round(1).to_string())
print("assimetria antes e depois do log:", round(e.km_por_treino.skew(), 2), "->", round(e.km_por_treino_log.skew(), 2))

treinos no arquivo: 1252488


-> dos elegíveis: 941195
-> na janela de 90 dias: 372012
-> ao ar livre: 199229 | GPS falhou: 0 | GPS ok sem distância ou zero: 0
-> com distância válida: 199229 | contas: 13308

contas sem treino com distância (ficam com 0 km): 691
  das quais têm treino ao ar livre na janela, todos sem distância válida (GPS falhou): 0

km por treino, quem tem distância: {'count': 13308.0, 'mean': 8.7, 'std': 9.5, 'min': 1.9, '25%': 5.0, '50%': 5.8, '75%': 7.6, 'max': 162.8}
km por treino por tipo principal (mediana, quem tem distância):
tipo_principal
caminhada    5.40
corrida      7.00
forca        5.20
funcional    5.20
pedal       27.30
yoga         5.10
assimetria antes e depois do log: 6.28 -> -0.17


## 3. Engajamento, plano e canal

In [4]:
ORD = {"possível churn": 0, "baixo": 1, "médio": 2, "alto": 3}
e["engajamento_ord"] = e.nivel_engajamento.map(ORD)
assert e.engajamento_ord.notna().all()
print(e.nivel_engajamento.value_counts().to_string())
e["plano_anual"] = (e.plano == "anual").astype(int)
print("\nplano anual:", int(e.plano_anual.sum()), f"({e.plano_anual.mean():.1%})")
canal = pd.get_dummies(e.canal_aquisicao, prefix="canal").astype(int)
e = pd.concat([e, canal], axis=1)
CANAIS = list(canal.columns)
print("\ncanal:"); print(e.canal_aquisicao.value_counts().to_string())

nivel_engajamento
alto              10689
médio              2366
possível churn      579
baixo               365

plano anual: 7727 (55.2%)

canal:
canal_aquisicao
organico                4410
influenciador           2379
meta_ads                2294
indicacao               2113
google_ads              1435
parceria_corporativa    1368


## 4. Matriz do k-means

In [5]:
NUM = ["treinos_por_semana", "dias_sem_treinar_log", "duracao_dias_log", "pct_ar_livre", "km_por_treino_log", "engajamento_ord"]
IND = ["plano_anual"] + CANAIS
print("vazios nas variáveis:", int(e[NUM + IND].isna().sum().sum()))
print("treinos_por_semana, assimetria:", round(e.treinos_por_semana.skew(), 2), "| máximo:", e.treinos_por_semana.max())
print("\ncorrelação entre as numéricas (elegíveis):")
print(e[NUM].corr().round(2))
sc = StandardScaler().fit(e[NUM])
Xn = pd.DataFrame(sc.transform(e[NUM]), columns=[c + "_z" for c in NUM])
X = pd.concat([Xn, e[IND].reset_index(drop=True)], axis=1)
print("\nmatriz:", X.shape)
print(X.describe().T[["mean", "std", "min", "max"]])

vazios nas variáveis: 0
treinos_por_semana, assimetria: 1.02 | máximo: 12.44

correlação entre as numéricas (elegíveis):
                      treinos_por_semana  dias_sem_treinar_log  duracao_dias_log  pct_ar_livre  km_por_treino_log  engajamento_ord
treinos_por_semana                  1.00                 -0.49              0.21         -0.03               0.15             0.42
dias_sem_treinar_log               -0.49                  1.00              0.10          0.08              -0.08            -0.83
duracao_dias_log                    0.21                  0.10              1.00         -0.01               0.04            -0.10
pct_ar_livre                       -0.03                  0.08             -0.01          1.00               0.51            -0.09
km_por_treino_log                   0.15                 -0.08              0.04          0.51               1.00             0.07
engajamento_ord                     0.42                 -0.83             -0.10         -0.0

## 5. k-means com k = 3

In [6]:
km3 = KMeans(n_clusters=K, n_init=10, random_state=SEED).fit(X)
lab = km3.labels_
print("inércia:", round(km3.inertia_, 1), "| voltas até convergir:", km3.n_iter_)
sil = silhouette_score(X, lab)
print("silhueta (todos os pontos):", round(sil, 3))
print("tamanho de cada grupo:", pd.Series(lab).value_counts().sort_index().to_dict(), "| soma:", len(lab))

inércia: 65108.5 | voltas até convergir: 13


silhueta (todos os pontos): 0.195
tamanho de cada grupo: {0: 4944, 1: 6256, 2: 2799} | soma: 13999


### Quanto cada variável separa os grupos

Para cada coluna da matriz, a parte da variação que fica **entre** os grupos (R², de 0 a 1). Perto de 1:
a variável praticamente define os grupos. Perto de 0: os grupos não diferem nela.

In [7]:
def r2(col):
    tot = ((col - col.mean()) ** 2).sum()
    ent = col.groupby(lab).apply(lambda s: len(s) * (s.mean() - col.mean()) ** 2).sum()
    return ent / tot if tot else np.nan
sep = pd.Series({c: r2(X[c]) for c in X.columns}).sort_values(ascending=False).rename("R2_entre_grupos")
print(sep.round(3).to_string())

pct_ar_livre_z               0.75
engajamento_ord_z            0.68
dias_sem_treinar_log_z       0.61
treinos_por_semana_z         0.19
km_por_treino_log_z          0.15
plano_anual                  0.06
duracao_dias_log_z           0.01
canal_meta_ads               0.00
canal_parceria_corporativa   0.00
canal_influenciador          0.00
canal_indicacao              0.00
canal_organico               0.00
canal_google_ads             0.00


### Diagnóstico: o canal pesa na distância sem separar os grupos

Os seis indicadores de canal têm R² zero, mas continuam somando distância entre quaisquer dois
assinantes de canais diferentes. Abaixo, a silhueta **dos mesmos grupos** medida sem as colunas de
canal, e um k-means com k = 3 rodado sem elas, para comparar. É só diagnóstico: o resultado oficial
continua sendo o das oito variáveis.

In [8]:
sem_canal = [c for c in X.columns if c not in CANAIS]
print("silhueta dos mesmos grupos, medida sem canal:", round(silhouette_score(X[sem_canal], lab), 3))
lab_sc = KMeans(n_clusters=K, n_init=10, random_state=SEED).fit_predict(X[sem_canal])
print("k-means k = 3 sem canal: silhueta", round(silhouette_score(X[sem_canal], lab_sc), 3), "| Rand ajustado contra o oficial:", round(adjusted_rand_score(lab, lab_sc), 3))

silhueta dos mesmos grupos, medida sem canal: 0.236


k-means k = 3 sem canal: silhueta 0.236 | Rand ajustado contra o oficial: 0.998


### Outras sementes e reamostragens

1. Mesma base, cinco sementes diferentes: os grupos mudam?
2. Cinco reamostragens com reposição: agrupa a amostra, aplica os centros nela a todos os elegíveis e
   compara com o original pelo índice de Rand ajustado (1 = idêntico, 0 = acaso). A aula pede acima de 0,9.

In [9]:
ari_seed = [adjusted_rand_score(lab, KMeans(n_clusters=K, n_init=10, random_state=s).fit_predict(X)) for s in [1, 2, 3, 4, 5]]
print("outra semente, Rand ajustado:", [round(v, 3) for v in ari_seed])
rng = np.random.default_rng(SEED)
ari_boot = []
for i in range(5):
    idx = rng.integers(0, len(X), len(X))
    kb = KMeans(n_clusters=K, n_init=10, random_state=SEED).fit(X.iloc[idx])
    ari_boot.append(adjusted_rand_score(lab, kb.predict(X)))
print("reamostragens, Rand ajustado:", [round(v, 3) for v in ari_boot], "| média:", round(np.mean(ari_boot), 3), "| mínimo:", round(min(ari_boot), 3))

outra semente, Rand ajustado: [1.0, 1.0, 1.0, 1.0, 1.0]


reamostragens, Rand ajustado: [0.983, 0.984, 0.988, 0.995, 0.975] | média: 0.985 | mínimo: 0.975


### Referência: inércia e silhueta de k = 2 a 8

O k = 3 é decisão do grupo; a curva fica aqui só para mostrar onde ele está. Silhueta numa amostra de
5.000 pontos (semente fixa) para k ≠ 3.

In [10]:
rows = []
for k in range(2, 9):
    mk = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(X)
    s = sil if k == K else silhouette_score(X, mk.labels_, sample_size=5000, random_state=SEED)
    rows.append({"k": k, "inercia": mk.inertia_, "silhueta": s, "menor_grupo": np.bincount(mk.labels_).min()})
curva = pd.DataFrame(rows)
curva["ganho_do_grupo_seguinte"] = -curva.inercia.diff().shift(-1)
print(curva.round(3).to_string(index=False))
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(curva.k, curva.inercia, "o-", color="#222"); ax[0].set_title("Inércia"); ax[0].set_xlabel("k")
ax[1].plot(curva.k, curva.silhueta, "o-", color="#222"); ax[1].set_title("Silhueta"); ax[1].set_xlabel("k")
for a_ in ax:
    a_.axvline(K, color="#e5484d", ls="--", lw=1)
ax[1].axhspan(0, 0.25, color="#ddd", alpha=.5); ax[1].axhspan(0.25, 0.5, color="#fbe3e3", alpha=.5)
fig.suptitle("k-means, 13.999 assinantes ativos que treinaram, 8 variáveis (k = 3 marcado)")
fig.tight_layout(); fig.savefig(S / "kmeans_k3_curvas.png", dpi=130)
curva.to_csv(S / "kmeans_k3_curva.csv", index=False, encoding="utf-8-sig")
print("salvo: saidas/kmeans_k3_curvas.png e saidas/kmeans_k3_curva.csv")

 k   inercia  silhueta  menor_grupo  ganho_do_grupo_seguinte
 2 76,457.04      0.27         3243                11,348.50
 3 65,108.54      0.20         2799                 7,022.30
 4 58,086.25      0.20         2007                 5,547.39
 5 52,538.86      0.21          700                 3,480.14
 6 49,058.72      0.20          567                 3,539.31
 7 45,519.41      0.20          689                 2,808.24
 8 42,711.17      0.20          686                      NaN
salvo: saidas/kmeans_k3_curvas.png e saidas/kmeans_k3_curva.csv


## 6. Perfil de cada grupo

Medianas na unidade original (não em z), tamanho, plano, canal e a lente de três fatores do grupo:
**ativos** (tamanho), **engajamento** (distribuição dos níveis) e **sinal de saída**. Como os dados param
em 30/09/2026, não há saída observada depois do corte; o sinal usado é a renovação automática
desligada no corte (`renovacao_desligada`), que o assinante ativo pode ter feito antes de sair.
Os grupos são numerados por tamanho (1 = maior).

In [11]:
ordem = pd.Series(lab).value_counts().index
mapa = {g: i + 1 for i, g in enumerate(ordem)}
e["grupo_k3"] = pd.Series(lab).map(mapa).values
g = e.groupby("grupo_k3")
perfil = pd.DataFrame({
    "assinantes": g.size(),
    "pct_base": g.size() / len(e) * 100,
    "treinos_por_semana": g.treinos_por_semana.median(),
    "dias_sem_treinar": g.dias_sem_treinar.median(),
    "meses_de_casa": g.meses_de_casa.median(),
    "pct_ar_livre": g.pct_ar_livre.median(),
    "km_por_treino": g.km_por_treino.median(),
    "pct_anual": g.plano_anual.mean() * 100,
    "pct_alto": g.nivel_engajamento.apply(lambda s: (s == "alto").mean() * 100),
    "pct_medio": g.nivel_engajamento.apply(lambda s: (s == "médio").mean() * 100),
    "pct_baixo": g.nivel_engajamento.apply(lambda s: (s == "baixo").mean() * 100),
    "pct_possivel_churn": g.nivel_engajamento.apply(lambda s: (s == "possível churn").mean() * 100),
    "pct_renovacao_desligada": g.renovacao_desligada.apply(lambda s: s.astype(float).mean() * 100),
})
base = pd.DataFrame({"assinantes": [len(e)], "pct_base": [100], "treinos_por_semana": [e.treinos_por_semana.median()],
    "dias_sem_treinar": [e.dias_sem_treinar.median()], "meses_de_casa": [e.meses_de_casa.median()], "pct_ar_livre": [e.pct_ar_livre.median()],
    "km_por_treino": [e.km_por_treino.median()], "pct_anual": [e.plano_anual.mean() * 100],
    "pct_alto": [(e.nivel_engajamento == "alto").mean() * 100], "pct_medio": [(e.nivel_engajamento == "médio").mean() * 100],
    "pct_baixo": [(e.nivel_engajamento == "baixo").mean() * 100], "pct_possivel_churn": [(e.nivel_engajamento == "possível churn").mean() * 100],
    "pct_renovacao_desligada": [e.renovacao_desligada.astype(float).mean() * 100]}, index=["base"])
perfil = pd.concat([perfil, base])
print(perfil.round(1).T)

                            1     2     3   base
assinantes               6256  4944  2799  13999
pct_base                44.70 35.30 20.00 100.00
treinos_por_semana       2.30  2.10  0.50   1.90
dias_sem_treinar         1.00  1.00 19.00   2.00
meses_de_casa            8.10  8.20  9.70   8.40
pct_ar_livre            80.60 14.30 78.30  71.40
km_por_treino            7.00  5.10  6.10   5.70
pct_anual               50.40 47.90 78.70  55.20
pct_alto                98.60 91.40  0.00  76.40
pct_medio                1.40  8.60 66.30  16.90
pct_baixo                0.00  0.00 13.00   2.60
pct_possivel_churn       0.00  0.00 20.70   4.10
pct_renovacao_desligada  4.60  5.10 51.80  14.20


### Canal de aquisição por grupo (% das linhas)

In [12]:
ct = pd.crosstab(e.grupo_k3, e.canal_aquisicao, normalize="index") * 100
ct.loc["base"] = e.canal_aquisicao.value_counts(normalize=True) * 100
print(ct.round(1))
print("\nplano por grupo (% das linhas):")
pl = pd.crosstab(e.grupo_k3, e.plano, normalize="index") * 100
pl.loc["base"] = e.plano.value_counts(normalize=True) * 100
print(pl.round(1))
print("\ntipo principal de treino por grupo (% das linhas):")
print((pd.crosstab(e.grupo_k3, e.tipo_principal, normalize="index") * 100).round(1))

canal_aquisicao  google_ads  indicacao  influenciador  meta_ads  organico  parceria_corporativa
grupo_k3                                                                                       
1                     10.20      15.50          17.50     16.10     31.20                  9.50
2                     10.20      14.90          16.90     15.70     31.70                 10.50
3                     10.30      14.50          16.00     18.30     31.90                  9.00
base                  10.30      15.10          17.00     16.40     31.50                  9.80

plano por grupo (% das linhas):
plano     anual  mensal
grupo_k3               
1         50.40   49.60
2         47.90   52.10
3         78.70   21.30
base      55.20   44.80

tipo principal de treino por grupo (% das linhas):
tipo_principal  caminhada  corrida  forca  funcional  pedal  yoga
grupo_k3                                                         
1                   19.00    63.50   0.00       0.00  17.50  0.

## 7. Salvar

In [13]:
out = e[["user_id", "grupo_k3"] + NUM + ["treinos_por_semana", "dias_sem_treinar", "duracao_dias", "meses_de_casa", "km_por_treino", "nivel_engajamento", "plano", "canal_aquisicao"]]
out = out.loc[:, ~out.columns.duplicated()]
out.to_parquet(P / "kmeans_k3_rotulos.parquet", index=False)
perfil.round(2).to_csv(S / "kmeans_k3_perfis.csv", encoding="utf-8-sig", index_label="grupo_k3")
ct.round(2).to_csv(S / "kmeans_k3_canal_por_grupo.csv", encoding="utf-8-sig", index_label="grupo_k3")
cent = pd.DataFrame(km3.cluster_centers_, columns=X.columns); cent.index = [mapa[i] for i in range(K)]
cent.sort_index().round(3).to_csv(S / "kmeans_k3_centros.csv", encoding="utf-8-sig", index_label="grupo_k3")
sep.round(4).to_csv(S / "kmeans_k3_separacao_por_variavel.csv", encoding="utf-8-sig", index_label="variavel")
print("rótulos:", out.shape, "-> dados/processados/kmeans_k3_rotulos.parquet")
print("saídas: kmeans_k3_perfis.csv, kmeans_k3_canal_por_grupo.csv, kmeans_k3_centros.csv, kmeans_k3_separacao_por_variavel.csv, kmeans_k3_curva.csv, kmeans_k3_curvas.png")

rótulos: (13999, 15) -> dados/processados/kmeans_k3_rotulos.parquet
saídas: kmeans_k3_perfis.csv, kmeans_k3_canal_por_grupo.csv, kmeans_k3_centros.csv, kmeans_k3_separacao_por_variavel.csv, kmeans_k3_curva.csv, kmeans_k3_curvas.png
